In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import os

In [ ]:
experiment = '12.09.2025_04.29_PM'
file = f'./sperm-data-analysis/experiments/{experiment}/final_data.csv'
df = pd.read_csv(file)

#remove controls
df = df[~df['fish'].str.contains('fishctrl|fishwater', case=False, regex=True, na=False)]

sample_map_int = {1:5, 2:6, 3:7, 4:8, 5:9, 6:10, 7:11, 8:12, 9:13, 10:14,
                  11:16, 12:17, 13:18, 14: 20, 15: 22, 16: 24, 17: 25, 18:26}

# Convert sample number to time
df['time'] = df['sample'].map(sample_map_int)

In [ ]:
# FOR FIGULRE 1 ONLY - Keep only sample 1
df_sample1 = df[df['sample'] == 1].copy()
print(f"\nNumber of fish with none_barcode data: {len(df_sample1)}")

In [ ]:
# Figure 1C: Number of unique barcodes per fish at first sampling timepoint
# AND Supp. Figure XX: Number of unique barcodes that EVER appeared for each fish

# Count unique barcodes per fish across all replicates
barcode_counts = (
    df.groupby('fish')['barcode']
    .nunique()
    .reset_index(name='unique_barcode_count')
)
# Build color dictionary
unique_fish = sorted(df['fish'].unique())
master_fish_order = [
    "fish1", "fish2", "fish4", "fish5", "fish8",
    "fish10", "fish12", "fish13", "fish14", "fish15",
    "fish16", "fish17", "fish18", "fish22", "fish26",
    "fish35", "fish37", "fish38", "fish40", 
    "fish42", "fish43", "fish44", "fish45", "fish50", "fish51" 
]
master_colors = [
    "#14466C", "#2278B5", "#add8e6", #blues
    "#1A702E", "#2FA148", "#9ED08A", #greens
    "#C4AA00", "#FFE116", #yellows
    "#940C07","#E62923", "#F69797", #reds
    "#673484", "#9268AD", "#C5B1D6", #purples
    "#CD6C18", "#F57F20", "#FCBB78", #oranges
    "#008080", "#43b3ae", "#96ded1", #teals
    "#CD2A91", "#E580BD", "#FCC7DC", #pinks
    "#282B3D", "#7C83BC" #periwinkles
]
color_map = dict(zip(master_fish_order, master_colors))

# Order fish numerically for legend
# Extract numeric part and sort using raw string
barcode_counts['fish_num'] = barcode_counts['fish'].str.extract(r'(\d+)').astype(int)
barcode_counts = barcode_counts.sort_values('fish_num')
# Create ordered categorical for proper legend ordering
fish_order = barcode_counts.sort_values('fish_num')['fish'].tolist()
barcode_counts['fish'] = pd.Categorical(barcode_counts['fish'], categories=fish_order, ordered=True)

mean_value = barcode_counts['unique_barcode_count'].mean()

# Plot figure
fig, ax = plt.subplots(figsize=(1.25, 2.5))
sns.swarmplot(
    data=barcode_counts,
    x=[0] * len(barcode_counts),
    y='unique_barcode_count',
    hue='fish',
    palette=color_map,
    size=6.5,
    alpha=0.8,
    ax=ax,
    dodge=False,
    hue_order=fish_order
)

# Add legend with proper ordering
ax.legend(
    title='Fish',
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=8,
    title_fontsize=9
)
# OR: remove legend 
# ax.legend_.remove()

ax.hlines(y=mean_value, xmin=-0.2, xmax=0.2, color='black', linewidth=1.5)
ax.text(0.25, mean_value, f"{mean_value:.2f}",
        va='center', ha='left', fontsize=9, color='black')
ax.set_xlim(-0.3, 0.5)
ax.set_ylim(0, 16)

# For t1 summary in main figure: Set y-axis ticks from 0 to 15
# For ALL TIME summary in supplement: Set y-axis ticks from 0 to 21
ax.set_yticks([0, 3, 6, 9, 12, 15, 18, 21])
ax.set_yticklabels([0, 3, 6, 9, 12, 15, 18, 21])

ax.set_ylabel("Mean barcode count across full lifespan", fontsize=10)
ax.set_xticks([])
ax.set_xlabel('')
for s in ['top', 'right', 'bottom']:
    ax.spines[s].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.plot([-0.3, 0.25], [0, 0], color='black', linewidth=1.5, clip_on=False)

plt.savefig("ColoredViolin_tALL_NOlegend.svg", bbox_inches='tight', transparent=True)
plt.savefig("ColoredViolin_tALL_NOlegend.png", dpi=600, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Calculate statistics on # barcodes per fish
counts = barcode_counts['unique_barcode_count']

stats = {
    'N':        len(counts),
    'Mean':     counts.mean(),
    'Median':   counts.median(),
    'Std Dev':  counts.std(),
    'Range':    f"{counts.min()} – {counts.max()}",
    'Min':      counts.min(),
    'Max':      counts.max(),
}

print("  Unique Barcodes per Fish — Summary")
print("=" * 38)
for k, v in stats.items():
    if isinstance(v, float):
        print(f"  {k:<12} {v:.3f}")
    else:
        print(f"  {k:<12} {v}")

In [ ]:
# Mixed Linear Model (by CRS)

# A) Count unique barcodes per fish at each timepoint
barcode_counts_time = (
    df.groupby(['fish', 'time'])['barcode']
    .nunique()
    .reset_index(name='unique_barcode_count')
)

# Order fish numerically for legend
barcode_counts_time['fish_num'] = barcode_counts_time['fish'].str.extract(r'(\d+)').astype(int)
fish_order = sorted(barcode_counts_time['fish'].unique(), key=lambda x: int(x.replace('fish', '')))
barcode_counts_time['fish'] = pd.Categorical(barcode_counts_time['fish'], categories=fish_order, ordered=True)

In [ ]:
# B) Function to compute Shannon diversity for a group (by CRS)
def shannon_diversity(group):
    counts = group['reads'].values
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    return -(p * np.log(p)).sum()

# Compute diversity per (fish, sample)
diversity = (
    df.groupby(['fish', 'time'])
    .apply(shannon_diversity, include_groups = False)
    .reset_index(name='diversity')
)

barcode_counts_time = barcode_counts_time.merge(diversity, on=['fish', 'time'])

In [ ]:
# C) Compute Stats (by CRS)
barcode_counts_time["time_centered"] = (
    barcode_counts_time.groupby("fish_num")["time"].transform(lambda x: x - x.mean())
)

# mask = barcode_counts_time.groupby("fish_num")["sample"].transform("count") >= 3
# df_filtered = barcode_counts_time[mask].copy()

import statsmodels.formula.api as smf
model = smf.mixedlm(
    "diversity ~ time_centered",
    data=barcode_counts_time,
    groups=barcode_counts_time["fish_num"],
    re_formula="~time_centered"   # random intercept
)
result = model.fit()
print(result.summary())

In [ ]:
# D) Figure 2D: Create full population MLM plot
fig, ax = plt.subplots(figsize=(3.1, 3.15))

# Options to toggle different elements
show_dots = False  # Set to True to show original scatter dots
show_grey_data_lines = False  # Set to False to hide grey background lines
show_individual_fits = True  # Set to False to hide individual fish fitted lines

# Plot dots (optional)
if show_dots:
    # Calculate centering offset for each sample
    offset_per_dot = 0.165
    sample_max_count = {}  # Track max number of stacked dots per sample
    
    # First pass: determine max offset per sample
    for sample in barcode_counts_time['time'].unique():
        sample_data = barcode_counts_time[barcode_counts_time['time'] == sample]
        max_stack = 0
        for count_value in sample_data['diversity'].unique():
            n_at_position = len(sample_data[sample_data['diversity'] == count_value])
            max_stack = max(max_stack, n_at_position)
        sample_max_count[sample] = max_stack
    
    # For each fish, stack overlapping values horizontally with centering
    for fish in fish_order:
        fish_data = barcode_counts_time[barcode_counts_time['fish'] == fish]
        
        first_point = True  # Track first point for legend
        for idx, row in fish_data.iterrows():
            sample = row['time']
            count = row['diversity']
            
            # Count how many fish already plotted at this exact (sample, count) position
            already_plotted = barcode_counts_time[
                (barcode_counts_time['time'] == sample) & 
                (barcode_counts_time['diversity'] == count) &
                (barcode_counts_time['fish_num'] < row['fish_num'])
            ]
            
            # Total fish at this position
            total_at_position = len(barcode_counts_time[
                (barcode_counts_time['time'] == sample) & 
                (barcode_counts_time['diversity'] == count)
            ])
            
            # Apply horizontal offset with centering
            position_in_stack = len(already_plotted)
            total_width = (total_at_position - 1) * offset_per_dot
            centering_offset = -total_width / 2
            horizontal_offset = centering_offset + (position_in_stack * offset_per_dot)
            
            ax.scatter(
                sample + horizontal_offset,
                count,
                color=color_map.get(fish, 'gray'),
                s=18,
                alpha=0.7,
                label=fish if first_point else "",  # Only label once per fish
                edgecolors='white',
                linewidth=0.3,
                zorder=2
            )
            first_point = False

# Plot grey background lines for each fish's actual data (optional)
if show_grey_data_lines:
    for fish in fish_order:
        fish_data = barcode_counts_time[barcode_counts_time['fish'] == fish].sort_values('time')
        ax.plot(
            fish_data['time'],
            fish_data['diversity'],
            color='lightgray',
            alpha=0.5,
            linewidth=1,
            zorder=1
        )

# Overlay colored fitted lines for each fish (optional)
if show_individual_fits:
    for fish, sub in barcode_counts_time.groupby("fish_num"):
        effects = result.random_effects[fish]
        ri = effects.get("Intercept", effects.get("Group", 0.0))
        rs = effects["time_centered"]
        group_mean = sub["time"].mean()
        x_vals = np.linspace(sub["time"].min(), sub["time"].max(), 100)
        x_centered = x_vals - group_mean
        y_pred = (intercept + ri) + (slope + rs) * x_centered
        ax.plot(
            x_vals, 
            y_pred, 
            color=color_map.get(f'fish{fish}', 'gray'),
            linewidth=1,
            alpha=0.9,
            label=f'fish{fish}' if not show_dots else "",  # Only label if dots aren't shown
            zorder=2
        )

# Add overall population trendline (bold black line)
# Calculate population-level prediction across the full time range
all_times = barcode_counts_time['time'].values
overall_mean = all_times.mean()
x_range = np.linspace(all_times.min(), all_times.max(), 100)
x_centered_overall = x_range - overall_mean
y_population = intercept + slope * x_centered_overall

ax.plot(
    x_range,
    y_population,
    color='black',
    linewidth=2,
    label='Population trend',
    zorder=3
)

# Format axes
ax.set_xlabel('Timepoint', fontsize=7)
ax.set_ylabel('Diversity', fontsize=7)
ax.set_ylim(0.5, 3.5)
ax.set_yticks([0.5, 1, 1.5, 2, 2.5, 3, 3.5])
ax.tick_params(axis='y', labelsize=7)

# Set x-axis to show all samples
all_samples = sorted(barcode_counts_time['time'].unique())
ax.set_xticks(all_samples)
ax.set_xticklabels(all_samples, fontsize=7)

# Add legend
#ax.legend(
 #   title='Fish',
  #  bbox_to_anchor=(1.05, 1),
   # loc='upper left',
    #frameon=False,
    #fontsize=7,
    #title_fontsize=7,
    #ncol=1
#)

# Clean up spines
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

plt.tight_layout()

#plt.savefig("Fig2_DiversityModel_v7_nolegend.svg", bbox_inches='tight', transparent=True)
#plt.savefig("Fig2_DiversityModel_v7_nolegend.png", dpi=600, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# E) Create alternative full population dot plot
fig, ax = plt.subplots(figsize=(4, 3))

# Calculate centering offset for each sample
offset_per_dot = 0.165
sample_max_count = {}  # Track max number of stacked dots per sample

# First pass: determine max offset per sample
for sample in barcode_counts_time['time'].unique():
    sample_data = barcode_counts_time[barcode_counts_time['time'] == sample]
    max_stack = 0
    for count_value in sample_data['diversity'].unique():
        n_at_position = len(sample_data[sample_data['diversity'] == count_value])
        max_stack = max(max_stack, n_at_position)
    sample_max_count[sample] = max_stack

# For each fish, stack overlapping values horizontally with centering
for fish in fish_order:
    fish_data = barcode_counts_time[barcode_counts_time['fish'] == fish]
    
    first_point = True  # Track first point for legend
    for idx, row in fish_data.iterrows():
        sample = row['time']
        count = row['diversity']
        
        # Count how many fish already plotted at this exact (sample, count) position
        already_plotted = barcode_counts_time[
            (barcode_counts_time['time'] == sample) & 
            (barcode_counts_time['diversity'] == count) &
            (barcode_counts_time['fish_num'] < row['fish_num'])
        ]
        
        # Total fish at this position
        total_at_position = len(barcode_counts_time[
            (barcode_counts_time['time'] == sample) & 
            (barcode_counts_time['diversity'] == count)
        ])
        
        # Apply horizontal offset with centering
        position_in_stack = len(already_plotted)
        total_width = (total_at_position - 1) * offset_per_dot
        centering_offset = -total_width / 2
        horizontal_offset = centering_offset + (position_in_stack * offset_per_dot)
        
        ax.scatter(
            sample + horizontal_offset,
            count,
            color=color_map.get(fish, 'gray'),
            s=18,
            alpha=0.7,
            label=fish if first_point else "",  # Only label once per fish
            edgecolors='white',
            linewidth=0.3
        )
        first_point = False

# Fixed effect coefficients
intercept = result.params["Intercept"]
slope = result.params["time_centered"]

# Overlay fitted line for each fish, restricted to its domain
for fish, sub in barcode_counts_time.groupby("fish_num"):
    effects = result.random_effects[fish]
    ri = effects.get("Intercept", effects.get("Group", 0.0))
    rs = effects["time_centered"]

    group_mean = sub["time"].mean()
    x_vals = np.linspace(sub["time"].min(), sub["time"].max(), 100)
    x_centered = x_vals - group_mean
    y_pred = (intercept + ri) + (slope + rs) * x_centered
    plt.plot(x_vals, y_pred, color = color_map.get(f'fish{fish}', 'gray'))

# Format axes
ax.set_xlabel('Timepoint', fontsize=8)
ax.set_ylabel('Diversity', fontsize=8)
ax.set_ylim(0, 4)
ax.set_yticks([0, 1, 2, 3, 4])
ax.tick_params(axis='y', labelsize=8)

# Set x-axis to show all samples (centered on actual sample values)
all_samples = sorted(df['time'].unique())
ax.set_xticks(all_samples)
ax.set_xticklabels(all_samples, fontsize=8)

# Add legend
ax.legend(
    title='Fish',
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=8,
    title_fontsize=8,
    ncol=1
)

# Clean up spines
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

plt.show()

In [ ]:
# F) Supplemental Figure XX: Plot each fish MLM on its own facet 
# dots at each sample timepoint to show number of unique barcodes

# Manually specify which fish to plot
fish_to_plot = [1, 2, 4, 5, 8, 10, 12,
               13, 14, 15, 16, 17, 18,
               22, 26, 35, 37, 38, 40,
               42, 43, 44, 45, 50, 51]  # Change this list to include whichever fish you want
# Set x-axis
all_samples = sorted(barcode_counts_time['time'].unique())
x_min = min(all_samples)
x_max = max(all_samples)

# Customization parameters
facet_spacing_h = 0.2 # h = horizontal
facet_spacing_v = 0.4  # v = vertical
label_pad = 2  # distance between axis labels and tick labels

# Options to toggle different elements
show_dots = True
show_grey_data_lines = False
show_individual_fits = True
show_population_trend = False

# Calculate grid dimensions
n_fish = len(fish_to_plot)
n_cols = 5
n_rows = (n_fish + n_cols - 1) // n_cols

# Create figure and subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.6*n_cols, 2.5*n_rows))
plt.subplots_adjust(hspace=facet_spacing_v, wspace=facet_spacing_h)

# Flatten axes array for easier iteration
axes = axes.flatten() if n_fish > 1 else [axes]

# Plot each fish
for idx, fish_num in enumerate(fish_to_plot):
    ax = axes[idx]
    fish_name = f'fish{fish_num}'
    fish_data = barcode_counts_time[barcode_counts_time['fish_num'] == fish_num].sort_values('time')
    
    # Plot dots
    if show_dots:
        for _, row in fish_data.iterrows():
            ax.scatter(
                row['time'],
                row['diversity'],
                color=color_map.get(fish_name, 'gray'),
                s=18,
                alpha=0.7,
                edgecolors='white',
                linewidth=0.3,
                zorder=2
            )
    
    # Plot grey background lines (optional)
    if show_grey_data_lines:
        ax.plot(
            fish_data['time'],
            fish_data['diversity'],
            color='lightgray',
            alpha=0.5,
            linewidth=1,
            zorder=1
        )
    
    # Plot individual MLM fitted line for each fish
    if show_individual_fits:
        effects = result.random_effects[fish_num]
        ri = effects.get("Intercept", effects.get("Group", 0.0))
        rs = effects["time_centered"]
        group_mean = fish_data["time"].mean()
        x_vals = np.linspace(fish_data["time"].min(), fish_data["time"].max(), 100)
        x_centered = x_vals - group_mean
        y_pred = (intercept + ri) + (slope + rs) * x_centered
        ax.plot(
            x_vals, 
            y_pred, 
            color=color_map.get(fish_name, 'gray'),
            linewidth=1.5,
            alpha=0.9,
            zorder=2
        )
    
    # Optional: Add overall population trendline for reference
    if show_population_trend:
        all_times = barcode_counts_time['time'].values
        overall_mean = all_times.mean()
        x_range = np.linspace(all_times.min(), all_times.max(), 100)
        x_centered_overall = x_range - overall_mean
        y_population = intercept + slope * x_centered_overall
        
        ax.plot(
            x_range,
            y_population,
            color='black',
            linewidth=2,
            linestyle='--',
            alpha=0.5,
            zorder=1
        )
    
    # Format title: "Fish 1"
    ax.set_title(f'Fish {fish_num}', fontsize=8, fontweight='bold', pad=5)
    
    # Format axes with label padding
    ax.set_xlabel('Timepoint', fontsize=7, labelpad=label_pad)
    ax.set_ylabel('Diversity', fontsize=7, labelpad=label_pad)
    ax.set_ylim(0.5, 3.5)
    ax.set_yticks([0.5, 1, 1.5, 2, 2.5, 3, 3.5])
    ax.tick_params(axis='both', labelsize=7)
    
    # Set x-axis
    all_samples = sorted(barcode_counts_time['time'].unique())
    ax.set_xticks(all_samples)
    ax.set_xticklabels(all_samples, fontsize=7)
    ax.set_xlim(x_min, x_max)
    
    # Clean up spines
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)

# Remove extra subplots if any
for idx in range(n_fish, len(axes)):
    fig.delaxes(axes[idx])

#plt.tight_layout()

plt.savefig("Fig2_DiversityModel_Facets.svg", bbox_inches='tight', transparent=True)
plt.savefig("Fig2_DiversityModel_Facets.png", dpi=600, bbox_inches='tight', transparent=True)
plt.show()